# Build hex-aggregated dispersal store

Distil the per-regime trajectory zarrs into the compact aggregate
store defined in `plans/hex_aggregate_store.md`. This notebook builds:

- `key.parquet` — one row per hex in the BSH model domain (geometry,
  area, water area, depth, coast distance, Fucus area, HELCOM
  subbasin). Built once from static inputs (BSH wet-region geojson,
  H0 grids, Fucus shapefile, HELCOM level-2 polygons). Land hexes
  are included with `water_area_m2 = 0` so the key covers every hex
  `hp.label` can assign to a trajectory position.
- `counts/regime=…/release_year=…/part.parquet` — one row per
  `(release_hex, age_bin, target_hex)` with `n_obs` aggregate. Built
  per regime × release_year from the zarrs.

Units are SI throughout (m², m). The HexProj configuration travels
as parquet file-level metadata so `hex_id` values can be rematerialised
into geometry downstream.

In [1]:
import json
import os
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import xarray as xr
from shapely.ops import unary_union

from hextraj import HexProj

from helpers import load_trajectories, mask_land_seeded

# Parameters

In [2]:
base_path = "/Users/wrath/src/github.com/geomar-od-lagrange/2025_fucus-dispersal"

# Unified hex grid (release + target), matching 024's Baltic origin.
hex_size_meters = 6000
hex_origin_lon = 18.0
hex_origin_lat = 59.0

# Age binning.
age_bin_days = 10        # 22 bins × 10 d = 220 d of drift.
output_dt_mins = 60      # zarr output cadence.

# Dev scope: one zarr per regime for release_year = 2019.
regimes = ["surface", "surface_stokes", "bottom"]
release_year = 2019

In [3]:
base_path = Path(base_path)
dataset_root = base_path / f"output/HexAggregates/r{hex_size_meters // 1000}km_v1"
dataset_root.mkdir(parents=True, exist_ok=True)

n_age_bins = 220 // age_bin_days   # 22
max_age_bin = n_age_bins - 1        # 21 (bins 0..21)

# Dask cluster

In [4]:
from dask.distributed import Client

scheduler_file = os.environ.get("SCHEDULER_FILE")
if scheduler_file:
    for _ in range(60):
        if os.path.exists(scheduler_file):
            break
        time.sleep(1)
    client = Client(scheduler_file=scheduler_file)
else:
    client = Client(ip="0.0.0.0")
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://10.136.64.206:8787/status,
Dashboard: http://10.136.64.206:8787/status,Workers: 4
Total threads: 12,Total memory: 36.00 GiB
Status: running,Using processes: True
Comm: tcp://10.136.64.206:64300,Workers: 0
Dashboard: http://10.136.64.206:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://10.136.64.206:64311,Total threads: 3
Dashboard: http://10.136.64.206:64312/status,Memory: 9.00 GiB
Nanny: tcp://10.136.64.206:64303,


# Hex projection + BSH-domain decomposition

In [5]:
hp = HexProj(
    projection_name="laea",
    lon_origin=hex_origin_lon,
    lat_origin=hex_origin_lat,
    hex_size_meters=hex_size_meters,
)

wet_gdf = gpd.read_file(base_path / "data/BSH_model_coastline/coastline.geojson")
always_wet_gdf = gpd.read_file(
    base_path / "data/BSH_model_coastline/coastline_always_wet.geojson"
)

wet_union = unary_union(wet_gdf.geometry)
always_wet_union = unary_union(always_wet_gdf.geometry)

# The key covers the full BSH model domain, not just the wet region.
# Label every fine+coarse H0 grid point; the union is every hex
# hp.label can return for any trajectory position that stays inside the
# model's lat/lon bounds. Wet vs. dry is captured later by water_area_m2.
_domain_ids = set()
for grid in ("fine", "coarse"):
    h0 = xr.open_dataset(
        base_path / f"min_data/bsh_operationalmodel_data/static_file_{grid}/H0_file_{grid}.nc"
    )
    lon2d, lat2d = np.meshgrid(h0.lon.values, h0.lat.values)
    labels = hp.label(lon2d.ravel(), lat2d.ravel())
    _domain_ids |= set(int(x) for x in labels if x >= 0)
hex_ids = np.asarray(sorted(_domain_ids), dtype=np.int32)
print(f"BSH-domain hexes: {len(hex_ids):,}")

BSH-domain hexes: 43,091


# Key file

## Hex geometries in EPSG:4326 and EPSG:3035 (equal-area for Europe)

In [6]:
hex_gdf = hp.to_geodataframe(hex_ids.tolist())
# to_geodataframe returns a 1-col GeoDataFrame indexed by hex_id; realign.
hex_gdf = hex_gdf.loc[hex_ids].reset_index().rename(columns={"index": "hex_id"})
hex_gdf["hex_id"] = hex_gdf["hex_id"].astype(np.int32)
hex_gdf = hex_gdf.set_crs(epsg=4326, allow_override=True)

hex_gdf_3035 = hex_gdf.to_crs(epsg=3035)
wet_union_3035 = gpd.GeoSeries([wet_union], crs=4326).to_crs(3035).iloc[0]
always_wet_union_3035 = (
    gpd.GeoSeries([always_wet_union], crs=4326).to_crs(3035).iloc[0]
)

## Areas (m², in EPSG:3035)

In [7]:
fucus_gdf = (
    gpd.read_file(base_path / "data/Fucus_location_shp/REDLIST_SIS_Macrophytes.shp")
    .loc[lambda df: df.F_vesiculo != 0, ["geometry"]]
    .to_crs(epsg=4326)
)
fucus_union_3035 = (
    gpd.GeoSeries([unary_union(fucus_gdf.geometry)], crs=4326).to_crs(3035).iloc[0]
)

hex_gdf["area_m2"] = hex_gdf_3035.geometry.area.astype(np.float32)
# Clip against prepared unions via a spatial index (shapely auto-prepares
# from geopandas ≥0.13). At ~50 k hexes × 2 unions the naive loop is fine.
hex_gdf["water_area_m2"] = hex_gdf_3035.geometry.intersection(
    wet_union_3035
).area.astype(np.float32)
hex_gdf["fucus_area_m2"] = hex_gdf_3035.geometry.intersection(
    fucus_union_3035
).area.astype(np.float32)

## Mean depth over always-wet cells (H0 > 0), fine-grid priority

In [8]:
def _h0_hex_frame(grid):
    h0 = xr.open_dataset(
        base_path / f"min_data/bsh_operationalmodel_data/static_file_{grid}/H0_file_{grid}.nc"
    )
    lon2d, lat2d = np.meshgrid(h0.lon.values, h0.lat.values)
    vals = h0.H0.values
    flat = pd.DataFrame({
        "lon": lon2d.ravel(),
        "lat": lat2d.ravel(),
        "H0": vals.ravel(),
    })
    flat = flat[(flat.H0 > 0) & np.isfinite(flat.H0)]
    flat["hex_id"] = hp.label(flat.lon.values, flat.lat.values)
    flat = flat[flat.hex_id >= 0]
    flat["grid"] = grid
    return flat[["hex_id", "grid", "H0"]]

h0_frame = pd.concat([_h0_hex_frame("fine"), _h0_hex_frame("coarse")], ignore_index=True)

# Fine-grid priority: where a hex has any fine cells, drop its coarse cells.
hex_has_fine = set(h0_frame.loc[h0_frame.grid == "fine", "hex_id"].unique())
mask = (h0_frame.grid == "fine") | (~h0_frame.hex_id.isin(hex_has_fine))
mean_depth = h0_frame[mask].groupby("hex_id")["H0"].mean()
hex_gdf["mean_depth_m"] = hex_gdf["hex_id"].map(mean_depth).astype(np.float32)
n_depth_nan = hex_gdf["mean_depth_m"].isna().sum()
print(f"hexes without H0 > 0 coverage (mean_depth_m NaN): {n_depth_nan}")

hexes without H0 > 0 coverage (mean_depth_m NaN): 31197


## Distance from centroid to always-wet coast (m, in EPSG:3035)

In [9]:
centroids_3035 = hex_gdf.geometry.centroid.to_crs(3035)
# GeoSeries.distance against a prepared boundary is fast enough at 50k points.
coast_boundary_3035 = always_wet_union_3035.boundary
hex_gdf["dist_to_coast_m"] = centroids_3035.distance(
    coast_boundary_3035
).astype(np.float32).values

/var/folders/w1/m9mm9h9167z_gcfzfffr0rgsh6j6kj/T/ipykernel_12994/2275106182.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_3035 = hex_gdf.geometry.centroid.to_crs(3035)


## HELCOM subbasin by centroid

In [10]:
subbasins = (
    gpd.read_file(
        base_path / "data/HELCOM_subbasins_2022_level2/HELCOM_subbasins_2022_level2.shp"
    )
    .to_crs(epsg=4326)
    .rename(columns={"level_2": "subbasin"})
    .reset_index(drop=True)
)
# Stable int8 lookup for the lookup JSON. -1 reserved for "outside".
subbasin_id_to_name = {-1: "_outside"}
for i, name in enumerate(subbasins["subbasin"].tolist()):
    subbasin_id_to_name[int(i)] = str(name)
subbasin_name_to_id = {v: k for k, v in subbasin_id_to_name.items() if k >= 0}

# Centroids only needed for the sjoin; build once here.
centroids_pts = gpd.GeoDataFrame(
    {"hex_id": hex_gdf["hex_id"].values},
    geometry=hex_gdf.geometry.centroid,
    crs=4326,
)
joined = gpd.sjoin(
    centroids_pts,
    subbasins[["geometry", "subbasin"]],
    how="left", predicate="within",
).drop_duplicates(subset="hex_id")
hex_gdf["helcom_subbasin"] = (
    joined.set_index("hex_id")
    .reindex(hex_gdf["hex_id"].values)["subbasin"]
    .map(subbasin_name_to_id)
    .fillna(-1)
    .astype(np.int8)
    .values
)
hex_gdf[["hex_id", "area_m2", "water_area_m2",
         "fucus_area_m2", "mean_depth_m", "dist_to_coast_m", "helcom_subbasin"]].head()

/Users/wrath/src/github.com/geomar-od-lagrange/2025_fucus-dispersal/.pixi/envs/default/lib/python3.14/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(
/var/folders/w1/m9mm9h9167z_gcfzfffr0rgsh6j6kj/T/ipykernel_12994/841043027.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  geometry=hex_gdf.geometry.centroid,


,hex_id,area_m2,water_area_m2,fucus_area_m2,mean_depth_m,dist_to_coast_m,helcom_subbasin
0,0,93530744.0,18166758.0,52885136.0,2.730000,1012.489197,-1
1,1,93530744.0,4507301.0,23311152.0,NaN,4390.951172,-1
2,2,93530744.0,59812392.0,66757968.0,15.160000,745.995544,14
3,3,93530744.0,56994044.0,93530744.0,7.242667,384.453186,-1
4,4,93530744.0,88804896.0,69004392.0,3.175250,4455.723145,14


## Serialise key.parquet as geoparquet with HexProj metadata

Write as proper geoparquet (geometry column owned by geopandas/pyarrow),
then extend the file-level metadata with the `hex_aggregate_store` JSON blob.

In [11]:
hex_proj_meta = {
    "projection_name": "laea",
    "lon_origin": hex_origin_lon,
    "lat_origin": hex_origin_lat,
    "hex_size_meters": hex_size_meters,
}
key_meta = {
    "hex_proj": hex_proj_meta,
    "area_crs": "EPSG:3035",
    "subbasin_id_to_name": subbasin_id_to_name,
}

# Drop lon_c/lat_c — centroids are derivable from geometry at query time.
key_gdf = hex_gdf[["hex_id", "geometry", "area_m2", "water_area_m2",
                    "fucus_area_m2", "mean_depth_m", "dist_to_coast_m",
                    "helcom_subbasin"]].copy()

key_path = dataset_root / "key.parquet"
key_gdf.to_parquet(key_path)

# Extend the file-level metadata with our custom JSON blob.
key_table = pq.read_table(key_path)
existing_meta = key_table.schema.metadata or {}
existing_meta[b"hex_aggregate_store"] = json.dumps(key_meta).encode("utf-8")
key_table = key_table.replace_schema_metadata(existing_meta)
pq.write_table(key_table, key_path, compression="zstd")
print(f"wrote {key_path} ({key_path.stat().st_size / 1e6:.2f} MB)")

# Expose key_df for validation cells.
key_df = key_gdf

wrote /Users/wrath/src/github.com/geomar-od-lagrange/2025_fucus-dispersal/output/HexAggregates/r6km_v1/key.parquet (2.42 MB)


# Per-regime counts

`hp.label` returns `-1` for NaN lon/lat, so masked land-seeded
trajectories naturally fall out when we filter `target_hex >= 0`.

In [12]:
def _zarr_for(regime):
    matches = sorted(
        (base_path / f"output/Trajectories/{regime}/{release_year}").glob("*.zarr")
    )
    assert len(matches) == 1, (regime, matches)
    return matches[0]


def build_counts(regime):
    zarr_path = _zarr_for(regime)
    # Filename convention: Fucus_BSH_YYYYMMDD_…
    fn_date = zarr_path.name.split("_")[2]
    assert fn_date.startswith(str(release_year)), (fn_date, release_year)

    stages = {}
    t0 = time.time()

    ds = xr.open_zarr(zarr_path)
    ds, _ = mask_land_seeded(ds)

    release_ts = pd.Timestamp(ds.time.isel(obs=0).compute().values[0])
    release_doy = int(release_ts.dayofyear)
    if release_doy == 366:
        print(f"[{regime}] leap-year release DOY 366; skipping per plan.")
        return None
    fn_doy = int(pd.Timestamp(fn_date).dayofyear)
    assert fn_doy == release_doy, (fn_doy, release_doy, regime)

    # Lazy (trajectory, obs) hex labels.
    target_hex = xr.apply_ufunc(
        hp.label, ds.lon, ds.lat,
        dask="parallelized", output_dtypes=[np.int64],
    )
    # Per-trajectory release_hex from obs=0 (NaN'd for land-seeded; filtered below).
    release_hex = xr.apply_ufunc(
        hp.label, ds.lon.isel(obs=0, drop=True), ds.lat.isel(obs=0, drop=True),
        dask="parallelized", output_dtypes=[np.int64],
    )

    # age_bin over obs indices → days → 10-day bins.
    obs_ages_days = ds.obs.values * output_dt_mins / (60 * 24)
    age_bin = (obs_ages_days // age_bin_days).astype(np.int32)
    age_bin_da = xr.DataArray(age_bin, dims=["obs"])

    frame = xr.Dataset({
        "target_hex": target_hex,
        "release_hex": release_hex,
        "age_bin": age_bin_da,
    }).to_dask_dataframe(dim_order=["trajectory", "obs"])

    # target_hex >= 0 filters both land-obs and land-seeded trajectories
    # (mask_land_seeded NaNs all lon/lat → label returns -1).
    t1 = time.time()
    valid_frame = frame[
        (frame.target_hex >= 0) & (frame.age_bin >= 0) & (frame.age_bin <= max_age_bin)
    ]
    counts = (
        valid_frame
        .groupby(["release_hex", "age_bin", "target_hex"])
        .size().rename("n_obs").reset_index().compute()
    )
    # n_valid: rows passing the filter, before groupby — equals sum(n_obs) by construction.
    n_valid = int(counts["n_obs"].sum())
    stages["compute_s"] = time.time() - t1

    # regime and release_year come back from the hive path on read —
    # storing them in data would just risk partition-schema drift.
    # release_doy varies within a single (regime, year) file in the
    # full build (73 doys per year), so it stays as a data column.
    counts["release_hex"] = counts["release_hex"].astype(np.int32)
    counts["target_hex"] = counts["target_hex"].astype(np.int32)
    counts["age_bin"] = counts["age_bin"].astype(np.int8)
    counts["n_obs"] = counts["n_obs"].astype(np.int32)
    counts["release_doy"] = np.int16(release_doy)

    # Write partition.
    t2 = time.time()
    part_dir = dataset_root / f"counts/regime={regime}/release_year={release_year}"
    part_dir.mkdir(parents=True, exist_ok=True)
    part_path = part_dir / "part.parquet"
    part_meta = {
        "hex_proj": hex_proj_meta,
        "age_bin_days": age_bin_days,
        "output_dt_mins": output_dt_mins,
        "regime": regime,
        "release_year": int(release_year),
        "release_doy": int(release_doy),
        "source_zarr": zarr_path.name,
    }
    tbl = pa.Table.from_pandas(counts, preserve_index=False)
    tbl = tbl.replace_schema_metadata({
        b"hex_aggregate_store": json.dumps(part_meta).encode("utf-8"),
    })
    pq.write_table(tbl, part_path, compression="zstd")
    stages["write_s"] = time.time() - t2

    stages["total_s"] = time.time() - t0
    return {
        "regime": regime,
        "zarr": zarr_path,
        "release_doy": release_doy,
        "n_valid": n_valid,
        "counts": counts,
        "part_path": part_path,
        "stages": stages,
    }

In [13]:
results = {}
for regime in regimes:
    print(f"\n=== {regime} ===")
    results[regime] = build_counts(regime)
    print(results[regime]["stages"])


=== surface ===


{'compute_s': 15.065140962600708, 'write_s': 0.12016010284423828, 'total_s': 16.68645691871643}

=== surface_stokes ===


{'compute_s': 13.951070785522461, 'write_s': 0.07222414016723633, 'total_s': 14.814594030380249}

=== bottom ===


{'compute_s': 12.599894046783447, 'write_s': 0.030786752700805664, 'total_s': 13.456563949584961}


# Validation

## Key-file stats

In [14]:
print(f"n hexes: {len(key_df):,}")
print("water_area_m2 min/median/max: "
      f"{key_df.water_area_m2.min():.0f} / "
      f"{key_df.water_area_m2.median():.0f} / "
      f"{key_df.water_area_m2.max():.0f}")
print(f"mean_depth_m NaN: {int(key_df.mean_depth_m.isna().sum())}")
print(f"fucus_area_m2 > 0: {int((key_df.fucus_area_m2 > 0).sum())}")
vc = key_df["helcom_subbasin"].value_counts().sort_index()
print("helcom_subbasin counts (id → n):")
for sid, n in vc.items():
    print(f"  {int(sid):>3} {subbasin_id_to_name[int(sid)]:<35s} {int(n):>6}")

n hexes: 43,091
water_area_m2 min/median/max: 0 / 0 / 93530744
mean_depth_m NaN: 31197
fucus_area_m2 > 0: 1839
helcom_subbasin counts (id → n):
   -1 _outside                             38626
    0 Åland Sea                              182
    1 Arkona Basin                           194
    2 Bay of Mecklenburg                      52
    3 Bornholm Basin                         446
    4 Bothnian Bay                           343
    5 Bothnian Sea                           637
    6 Eastern Gotland Basin                  805
    7 Gdansk Basin                            64
    8 Gulf of Finland                        319
    9 Gulf of Riga                           200
   10 Kattegat                               258
   11 Kiel Bay                                37
   12 Northern Baltic Proper                 351
   13 The Quark                               86
   14 Western Gotland Basin                  367
   15 Great Belt                             115
   16 The Sound        

## Per-regime counts stats

In [15]:
for regime, r in results.items():
    if r is None:
        print(f"[{regime}] skipped")
        continue
    counts = r["counts"]
    print(f"\n[{regime}]")
    print(f"  rows: {len(counts):,}")
    print(f"  sum(n_obs): {int(counts.n_obs.sum()):,}")
    print(f"  stage seconds: {r['stages']}")


[surface]
  rows: 2,467,933
  sum(n_obs): 299,615,940
  stage seconds: {'compute_s': 15.065140962600708, 'write_s': 0.12016010284423828, 'total_s': 16.68645691871643}

[surface_stokes]
  rows: 1,482,651
  sum(n_obs): 302,300,406
  stage seconds: {'compute_s': 13.951070785522461, 'write_s': 0.07222414016723633, 'total_s': 14.814594030380249}

[bottom]
  rows: 608,758
  sum(n_obs): 299,431,350
  stage seconds: {'compute_s': 12.599894046783447, 'write_s': 0.030786752700805664, 'total_s': 13.456563949584961}


## Key-completeness invariant

Every `release_hex` and `target_hex` in counts must be in `key.parquet`'s
`hex_id` set. The key is built from the full BSH-domain grid; any
trajectory that stays within the model's lat/lon bounds necessarily
labels to a hex already in the key. If this assert ever fires,
investigate — don't paper over it by extending the key after the fact.

In [16]:
key_ids = set(int(x) for x in key_df["hex_id"].values)
for regime, r in results.items():
    if r is None:
        continue
    unseen = (set(int(x) for x in r["counts"]["release_hex"].values)
              | set(int(x) for x in r["counts"]["target_hex"].values)) - key_ids
    assert not unseen, (regime, sorted(unseen))
print("PASS: every release_hex and target_hex is in key.parquet.")

PASS: every release_hex and target_hex is in key.parquet.


## Conservation cross-check: sum(n_obs) == non-masked (traj, obs) pairs

`n_valid` is captured inside `build_counts` immediately after the single-pass
groupby, so no re-scan is needed here. The check: for each regime, what went
into the groupby equals what came out.

In [17]:
for regime, r in results.items():
    if r is None:
        print(f"[{regime}] skipped")
        continue
    n_valid = r["n_valid"]
    n_stored = int(r["counts"]["n_obs"].sum())
    print(f"[{regime}] n_valid={n_valid:,}  stored sum(n_obs)={n_stored:,}")
    assert n_valid == n_stored, (regime, n_valid, n_stored)
print("PASS: conservation holds for all regimes.")

[surface] n_valid=299,615,940  stored sum(n_obs)=299,615,940
[surface_stokes] n_valid=302,300,406  stored sum(n_obs)=302,300,406
[bottom] n_valid=299,431,350  stored sum(n_obs)=299,431,350
PASS: conservation holds for all regimes.


## Output sizes

In [18]:
total = 0
entries = [("key.parquet", key_path)]
for regime in regimes:
    r = results.get(regime)
    if r is None:
        continue
    entries.append((f"counts/{regime}/release_year={release_year}", r["part_path"]))

for label, p in entries:
    size = p.stat().st_size
    total += size
    print(f"  {label:<60s} {size/1e6:>9.2f} MB")
print(f"  {'TOTAL':<60s} {total/1e6:>9.2f} MB")

  key.parquet                                                       2.42 MB
  counts/surface/release_year=2019                                  7.74 MB
  counts/surface_stokes/release_year=2019                           4.83 MB
  counts/bottom/release_year=2019                                   2.27 MB
  TOTAL                                                            17.26 MB
